In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder

# ── Load original ──
df = pd.read_csv("donors(opal).csv")
df = df.drop(columns=['donor_id', 'first_name', 'last_name'])

# ── 1. Gender Encoding ──
df['gender'] = df['gender'].str.strip().str.lower().replace({'male':'Male','m':'Male','female':'Female','f':'Female'})
df = pd.get_dummies(df, columns=['gender'], drop_first=True)

# ── 2. Blood Type Encoding ──
df['blood_type'] = df['blood_type'].fillna('Unknown').str.strip().str.upper()
valid_blood_types = ['A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-']
df.loc[~df['blood_type'].isin(valid_blood_types), 'blood_type'] = 'Unknown'
df = pd.get_dummies(df, columns=['blood_type'], prefix='blood', drop_first=False)

# ── 3. Organs Encoding ──
normalization_map = {'Kidneys':'Kidney','Corneas':'Cornea','Lungs':'Lung','Platelets':'Platelet'}
def clean_organs(organ_str):
    if pd.isna(organ_str): return []
    items = [i.strip() for i in str(organ_str).split(';')]
    return list(set([normalization_map.get(i, i) for i in items]))

df['organs_list'] = df['organs_donating'].apply(clean_organs)
mlb = MultiLabelBinarizer()
one_hot_df = pd.DataFrame(mlb.fit_transform(df['organs_list']), columns=mlb.classes_, index=df.index)
df = pd.concat([df.drop(columns=['organs_list']), one_hot_df], axis=1)

# ── 4. Medical Conditions Encoding ──
def clean_medical_conditions(c):
    if pd.isna(c) or str(c).lower() == 'none': return []
    return list(set([x.strip().title() for x in str(c).split(';')]))

df['conditions_list'] = df['medical_conditions'].apply(clean_medical_conditions)
mlb_conditions = MultiLabelBinarizer()
conditions_encoded = mlb_conditions.fit_transform(df['conditions_list'])
condition_columns = [f"Condition_{col.replace(' ','_')}" for col in mlb_conditions.classes_]
conditions_df = pd.DataFrame(conditions_encoded, columns=condition_columns, index=df.index)
df = pd.concat([df.drop(columns=['conditions_list']), conditions_df], axis=1)

# ── 5. Hepatitis Encoding ──
df['hepatitis_status'] = df['hepatitis_status'].map({'Negative':0,'Positive':1,'Unknown':0})

# ── 6. Alive/Deceased Encoding ──
df['alive/deceased'] = df['alive/deceased'].map({'Alive':1,'Deceased':0})

# ── 7. Time Features ──
df['time_of_death'] = pd.to_datetime(df['time_of_death'], errors='coerce')
df['death_hour']  = df['time_of_death'].dt.hour.fillna(-1).astype(int)
df['death_day']   = df['time_of_death'].dt.day.fillna(-1).astype(int)
df['death_month'] = df['time_of_death'].dt.month.fillna(-1).astype(int)

# ── 8. Harvest Interval ──
df['time_harvested'] = pd.to_datetime(df['time_harvested'], errors='coerce')
df['harvest_time_hours'] = (df['time_harvested'] - df['time_of_death']).dt.total_seconds() / 3600
df['harvest_time_hours'] = df['harvest_time_hours'].fillna(-1)

# ── 9. Cause of Death Encoding ──
def group_cause_of_death(cause):
    if pd.isna(cause): return 'Not_Applicable'
    cause = str(cause).lower()
    if any(w in cause for w in ['heart','cardiac','infarction','stroke','cardiomyopathy','arrhythmia']): return 'Cardiovascular'
    elif any(w in cause for w in ['brain','cerebral','aneurysm','intracranial','hemorrhage','encephalitis']): return 'Neurological'
    elif any(w in cause for w in ['accident','injury','trauma','drowning']): return 'Accident_Trauma'
    elif any(w in cause for w in ['failure','insufficiency','cirrhosis','nephropathy']): return 'Organ_Failure'
    elif any(w in cause for w in ['respiratory','pulmonary','copd','fibrosis','ards']): return 'Respiratory'
    elif any(w in cause for w in ['sepsis','septic','shock']): return 'Infection_Sepsis'
    elif any(w in cause for w in ['cancer','leukemia','tumor']): return 'Cancer'
    elif any(w in cause for w in ['diabetic','ketoacidosis']): return 'Metabolic'
    else: return 'Other_Unknown'

df['cause_category'] = df['cause_of_death'].apply(group_cause_of_death)
cause_encoded = pd.get_dummies(df['cause_category'], prefix='Cause')
df = pd.concat([df, cause_encoded], axis=1)

# ── 10. Location Encoding ──
city_dummies = pd.get_dummies(df['city'], prefix='City')
le = LabelEncoder()
df['hospital_encoded'] = le.fit_transform(df['hospital_name'].astype(str))
df = pd.concat([df, city_dummies], axis=1)
df = df.drop(columns=['city', 'hospital_name'])

# ── 11. Drop remaining raw columns ──
df = df.drop(columns=['contact_number'], errors='ignore')

# ── Save final merged file ──
df.to_csv("FINAL_MERGED_ALL_ENCODED.csv", index=False)
df.to_excel("FINAL_MERGED_ALL_ENCODED.xlsx", index=False)

print(f"✅ Done! Shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist())

✅ Done! Shape: (800, 68)
Total columns: 68
['age', 'organs_donating', 'medical_conditions', 'hepatitis_status', 'alive/deceased', 'time_of_death', 'cause_of_death', 'time_harvested', 'gender_Male', 'blood_A+', 'blood_A-', 'blood_AB+', 'blood_AB-', 'blood_B+', 'blood_B-', 'blood_O+', 'blood_O-', 'Blood', 'Bone Marrow', 'Cornea', 'Heart', 'Kidney', 'Liver', 'Lung', 'Pancreas', 'Plasma', 'Platelet', 'Skin', 'Condition_Asthma', 'Condition_Diabetes', 'Condition_Heart_Disease', 'Condition_Hypertension', 'death_hour', 'death_day', 'death_month', 'harvest_time_hours', 'cause_category', 'Cause_Accident_Trauma', 'Cause_Cancer', 'Cause_Cardiovascular', 'Cause_Infection_Sepsis', 'Cause_Metabolic', 'Cause_Neurological', 'Cause_Not_Applicable', 'Cause_Organ_Failure', 'Cause_Other_Unknown', 'Cause_Respiratory', 'hospital_encoded', 'City_Abbottabad', 'City_Bahawalpur', 'City_Dera Ghazi Khan', 'City_Faisalabad', 'City_Gujranwala', 'City_Hyderabad', 'City_Islamabad', 'City_Jhelum', 'City_Karachi', 'City

In [3]:
import os
print(os.getcwd())

d:\OPAL AI ALL MATERIAL\dataset eda
